In [ ]:
import requests 
import json 
from dotenv import load_dotenv
import os
import pandas as pd
import time
from aws_upload import upload_file_to_s3

load_dotenv()

API_KEY = os.getenv('AZURE_API_KEY')
AWS_PREFIX = "https://data-visualization-benchmark.s3.us-west-2.amazonaws.com"

class ResponseExtractor:
    
    def __init__(
        self,
        test_type, 
        prompt_type="indist_instructions_question", 
        top_p_dir='p04', 
        temperature_dir='t1',
        agentType='GPT-4V'
    ):  
        
        self.questions = pd.read_csv(f'{AWS_PREFIX}/{test_type}/questions.csv')
        self.top_p_dir = top_p_dir
        self.temperature_dir = temperature_dir
        self.test_type = test_type
        self.prompt_type = prompt_type

        # model_response_url = f'{AWS_PREFIX}/{test_type}/responses/{prompt_type}/{top_p_dir}/{temperature_dir}/model_responses.csv'
        model_response_url = f'{AWS_PREFIX}/{test_type}/responses/{prompt_type}/{top_p_dir}/{temperature_dir}/{agentType.replace("/", "-")}/model_responses.csv'
        print(model_response_url)
        self.model_responses = pd.read_csv(model_response_url)
        self.model_responses = self.model_responses[self.model_responses["agentType"] == agentType]
        # self.model_responses = self.model_responses.drop_duplicates(subset=['agentType', 'image_file', 'question'])
        print(len(self.model_responses), "responses found for", agentType)
        self.model_responses = self.model_responses.merge(self.questions[['question', 'image_file']])

        api_base = os.getenv('AZURE_API_BASE')
        deployment_name = "vlm-datavis-grader"
        base_url = f"{api_base}/openai/deployments/{deployment_name}"

        self.headers = {   
            "Content-Type": "application/json",   
            "api-key": API_KEY 
        } 
        self.endpoint = f"{base_url}/chat/completions?api-version=2023-12-01-preview" 

    def create_question_prompt(self, question, agentResponse, choices=""):
        task_description = "Please read the following example. Then extract the answer from the model response and type it at the end of the prompt."
        if len(choices) == 0:
            if self.test_type == "chartqa-test-categorical":
                hint = "Hint: Please answer the question requiring in one or two words."
            else:
                hint = f"Hint: Please answer the question requiring a floating-point number with two decimal places and provide the final value, e.g., 1.23, 1.34, 1.45, at the end."
        else:
            hint = "Hint: Please answer the question and provide the correct option."
        question = f"Question: {question}"
        choices = f"Choices: {choices}\n" if choices else ""
        model_responses = f"Model response: {agentResponse}"
        end = "Extracted answer:"
        return f"{task_description}\n{hint}\n{question}\n{model_responses}\n{end}"

    def gpt4_find_answer(self, question, agentResponse, choices):
        prompt = self.create_question_prompt(question, agentResponse, choices)
        data = { 
            "messages": [ 
            { "role": "system", "content": "You are a helpful assistant." }, 
            { "role": "user", "content": [  
                { 
                "type": "text",
                "text": prompt
                }
            ] 
            } 
            ],
            "max_tokens": 2000 
        }
        print(data)

        response = requests.post(self.endpoint, headers=self.headers, json=data)
        if response.status_code == 200:
            response = response.json()
            try:
                model_res = response['choices'][0]["message"]["content"]
                print(model_res)
                return model_res
            except:
                print("Error in response, agent Response:", agentResponse)
                return "RAW: " + agentResponse
        if response.status_code == 429:
            print("Rate limit exceeded")
        if response.status_code == 404:
            print("Wrong endpoint")

        return None

    def build_multiple_choice_string(self, response):
        row = self.questions[
                (response["question"] == self.questions["question"]) & 
                (response["image_file"] == self.questions["image_file"])
        ].iloc[0]

        textInput = row["text_input_type"]

        if textInput ==  "multiple_choice_question":
            row = self.questions[
                (response["question"] == self.questions["question"]) & 
                (response["image_file"] == self.questions["image_file"])
            ].iloc[0]

            mc1, mc2, mc3, mc4 = row["mc1"], row["mc2"], row["mc3"], row["mc4"]
            multipleChoice = []
            for mc in [mc4,mc3,mc2,mc1]:
                if mc and mc != "" and mc != "Skip":
                    multipleChoice.append(mc)
            mcString = ", ".join(multipleChoice)
        else:
            mcString = ""
        
        return mcString

    def extract_response(self):
        # question_meta = self.questions
        answers_df = pd.DataFrame(columns=list(self.model_responses.columns) + ["agent_response"])

        count = 0
        responses = []
        for i, response in self.model_responses.iterrows():
            
            mcString = self.build_multiple_choice_string(response)

            question = response["question"]
            agent_response = response["agent_response"]
            answer = self.gpt4_find_answer(question, agent_response, mcString)

            response_row = response
            response_row["agent_response"] = answer
            if (answer == None):
                break

            answers_df.loc[len(answers_df)] = response_row

            time.sleep(3)
            count += 1
            print("COUNT", count)

        return answers_df


def gpt4_extract():
    model_configs = [
        {'model': "GPT-4V",  'top_p': 'p1', 'temp': 't02'},
        {'model': "liuhaotian/llava-v1.6-34b",  'top_p': 'p1', 'temp': 't04'},
        {'model': "Salesforce/blip2-flan-t5-xxl",  'top_p': 'p1', 'temp': 't10'},
        {'model': "Salesforce/blip2-flan-t5-xl",  'top_p': 'p06', 'temp': 't1'}, 
        {'model': "llava-hf/llava-1.5-7b-hf",  'top_p': 'p04', 'temp': 't1'},
        {'model': "llava-hf/llava-1.5-13b-hf",  'top_p': 'p1', 'temp': 't04'},
        {'model': "google/pix2struct-chartqa-base",  'top_p': 'p08', 'temp': 't1'},
        {'model': "google/matcha-chartqa",  'top_p': 'p04', 'temp': 't1'},
    ]
    prompt_type="indist_instructions_question"

    
    for test_type in [
        # "ggr", 
        # 'calvi-standard',
        # 'calvi-trick',
        # 'holf2',
        # "holf",
        # 'chartqa-test-continuous',
        'chartqa-test-continuous-human'
        # "vlat",
    ]:
        for model_config in model_configs:
            temperature_dir = model_config['temp']
            top_p_dir = model_config['top_p']
            model = model_config['model']
            _dir = f'responses/{prompt_type}/{top_p_dir}/{temperature_dir}/{model.replace("/", "-")}'
            local_dir = f"./{_dir}"
            if not os.path.exists(_dir):
                os.makedirs(local_dir)
            print(local_dir)
            
            response_extractor = ResponseExtractor(
                test_type=test_type, 
                prompt_type="indist_instructions_question",
                agentType=model,
                top_p_dir=top_p_dir, 
                temperature_dir=temperature_dir,
            )
            answers_df = response_extractor.extract_response()
            file_name = f"./{_dir}/{test_type}_extracted_responses.csv"
            print(file_name)
            answers_df.to_csv(file_name)
            upload_file_to_s3(file_name, test_type, f'{_dir}/extracted_responses.csv')